# Notebook 06: Landscape-Oriented Modelling of Urban Surface Warming (Hotspots)

**Purpose:** Examine which physiographic landscape elements — climate zone, geology, and topographic elevation — predict (A1) the *spatial extent* and (A2) the *thermal intensity* of persistent urban heat island hotspots across Nigerian cities.

**Study Year:** 2024 | **Cities:** 50 AOIs from Notebook 01 | **Hotspot data:** Enriched polygons from Notebook 05  
**Configuration:** `config.py` | **Functions:** `src/modelling_utils.py`

---

<a id="table-of-contents"></a>

## Table of Contents

1. [Setup and Data Loading](#section-1)
   1. [Environment Setup](#section-1-1)
   2. [Load Data Sources](#section-1-2)
2. [Dataset Construction](#section-2)
   1. [Polygon-Level Continuous Dataset (Analyses 1 & 2)](#section-2-2)
   2. [Data Composition — Donut Chart (Day | Night)](#section-2-3)
   3. [Data Composition — Sankey Diagram (Day / Night)](#section-2-4)
   4. [Skewness Diagnostic — Hotspot Extent (Area)](#section-2-5)
   5. [Skewness Diagnostic — Hotspot Intensity (Mean LST)](#section-2-6)
3. [Analysis 1 — Hotspot Extent (OLS with Clustered SEs + Wild Bootstrap)](#section-3)
   1. [Day & Night Models](#section-3-1)
   2. [Variance Partitioning (Day | Night)](#section-3-2)
   3. [Coefficient Forest Plots (Day | Night)](#section-3-3)
4. [Analysis 2 — Hotspot Intensity (OLS with Clustered SEs + Wild Bootstrap)](#section-4)
   1. [Day & Night Models](#section-4-1)
   2. [Variance Partitioning (Day | Night)](#section-4-2)
   3. [Coefficient Forest Plots (Day | Night)](#section-4-3)
5. [Summary and Export](#section-5)

---

## Code Organisation

| Module | Key functions used |
|--------|-------------------|
| `config.py` | `MODEL_OUTPUT_DIR`, `HIERARCHY_BLOCKS`, `WILD_BOOTSTRAP_CONFIG`, `OLS_CLUSTERED_CONFIG`, `OLS_INTENSITY_CONFIG`, `MODEL_EXPORT_CONFIG`, `OUTPUT_DIR`, `LANDSCAPE_DATA_PATH`, `LANDSCAPE_CLASS_ATTRIBUTES`, `load_cities_from_daily_analysis()` |
| `src/modelling_utils.py` | `load_aoi_boundaries()`, `extract_dominant_landscape()`, `build_binary_hotspot_dataset()`, `fit_hierarchical_ols_clustered()`, `wild_cluster_bootstrap()`, `format_ols_coefficients()`, `plot_variance_partitioning_panel()`, `plot_coefficient_forest_panel()`, `plot_data_composition_donut()`, `plot_data_composition_sankey()`, `plot_skewness_diagnostic()`, `save_model_metadata()` |
| `src/data_io.py` | `load_landscape_data()` |

---

## Methodology

### Hierarchical Block Entry

Both analyses use **hierarchical (sequential) regression** where physiographic landscape elements are entered as cumulative blocks in a theoretically motivated order:

| Block | Variable | Rationale |
|-------|----------|----------|
| 1 | Climate Zone | Macro-scale thermal regime; strongest control on base LST |
| 2 | + Geology | Parent material affects albedo, thermal inertia, and drainage |
| 3 | + Elevation | Lapse-rate modulation and local air circulation |

### Analysis 1 — Hotspot Extent (OLS with Clustered SEs)

**DV:** `log(area_km2)` — log-transformed spatial extent of each P90 hotspot polygon.

**Method:** Hierarchical OLS with city-clustered standard errors and Wild Cluster Bootstrap *p*-values.

**Goodness-of-fit:** Ordinary R² — proportion of variance explained.

The OLS model:

$$\ln(\text{area}_{ij}) = \beta_0 + \sum_k \beta_k X_{ijk} + \varepsilon_{ij}$$

where $i$ indexes polygons and $j$ indexes cities. Standard errors are clustered at the city level.

**Block significance:** Partial F-test between successive models:

$$F = \frac{(SSR_{\text{restricted}} - SSR_{\text{full}}) / q}{SSR_{\text{full}} / (n - k - 1)}$$

### Analysis 2 — Hotspot Intensity (OLS with Clustered SEs)

**DV:** Mean LST within each P90 hotspot polygon (Day: `LST_day_mean`, Night: `LST_night_mean`).

**Method:** Identical to Analysis 1 — hierarchical OLS with city-clustered SEs and Wild Cluster Bootstrap *p*-values.

**Rationale:** Tests whether landscape composition predicts how *hot* a hotspot is, complementing the extent analysis.

### Wild Cluster Bootstrap (all OLS models)

With ~30–40 distinct city clusters, conventional cluster-robust standard errors may over-reject the null hypothesis (Cameron, Gelbach & Miller, 2008). The **Wild Cluster Bootstrap** provides a more conservative alternative:

**Algorithm** (Cameron, Gelbach & Miller, 2008):

1. Fit the unrestricted OLS model; store coefficients $\hat{\beta}$ and residuals $\hat{e}_i$.
2. For $b = 1, \ldots, B$ ($B = 999$):
   - Draw one **Rademacher weight** $v_g \in \{-1, +1\}$ (equal probability) per cluster $g$.
   - Construct bootstrap DV: $y^*_i = X_i \hat{\beta} + v_{g(i)} \cdot \hat{e}_i$.
   - Fit OLS on $(X, y^*)$; compute cluster-robust $t$-statistic $t^*_b$ for each coefficient.
3. **Bootstrap *p*-value**: $p_{\text{boot}} = \frac{1 + \#\{|t^*_b| \geq |t_{\text{obs}}|\}}{1 + B}$

The Rademacher distribution is preferred over the standard normal because it preserves the kurtosis of the original residuals and produces more reliable inference with few clusters (Webb, 2022).

**Reference:** Cameron, A.C., Gelbach, J.B. & Miller, D.L. (2008). Bootstrap-based improvements for inference with clustered errors. *Review of Economics and Statistics*, 90(3), 414–427.

<a id="section-1"></a>

## 1. Setup and Data Loading

<a id="section-1-1"></a>

### 1.1 Environment Setup

In [ ]:
# ── Standard Libraries ───────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json, importlib

# ── Statistical Modelling Libraries ───────────────────────────────────
import statsmodels.api as sm
from scipy import stats as scipy_stats

# ── Project path setup ─────────────────────────────────────────────
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# ── Config (reload to pick up latest changes) ───────────────────────
import config
importlib.reload(config)

from config import (
    OUTPUT_DIR, PROJECT_ROOT,
    LANDSCAPE_DATA_PATH, LANDSCAPE_CLASS_ATTRIBUTES,
    MODEL_OUTPUT_DIR, HIERARCHY_BLOCKS,
    WILD_BOOTSTRAP_CONFIG,
    OLS_CLUSTERED_CONFIG, OLS_INTENSITY_CONFIG,
    MODEL_EXPORT_CONFIG,
    load_cities_from_daily_analysis, STUDY_YEAR,
)

# ── Custom modules (reload for development) ─────────────────────────
import data_io;           importlib.reload(data_io)
import modelling_utils;   importlib.reload(modelling_utils)

from data_io import load_landscape_data
from modelling_utils import (
    load_aoi_boundaries,
    extract_dominant_landscape,
    build_binary_hotspot_dataset,
    fit_hierarchical_ols_clustered,
    wild_cluster_bootstrap,
    format_ols_coefficients,
    plot_variance_partitioning_panel,
    plot_coefficient_forest_panel,
    plot_data_composition_donut,
    plot_data_composition_sankey,
    plot_skewness_diagnostic,
    save_model_metadata,
)

# ── Display settings ─────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = MODEL_EXPORT_CONFIG.get('figure_dpi', 300)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10

# ── Output directory setup ───────────────────────────────────────────
TABLES_DIR = MODEL_OUTPUT_DIR / MODEL_EXPORT_CONFIG['tables_subdir']
FIGURES_DIR = MODEL_OUTPUT_DIR / MODEL_EXPORT_CONFIG['figures_subdir']
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Wild Cluster Bootstrap settings ──────────────────────────────────
RUN_BOOTSTRAP = WILD_BOOTSTRAP_CONFIG.get('enabled', True)
N_BOOTSTRAP   = WILD_BOOTSTRAP_CONFIG.get('n_bootstrap', 999)
BOOT_SEED     = WILD_BOOTSTRAP_CONFIG.get('random_state', 42)

# ── Configuration summary ────────────────────────────────────────────
print("=" * 70)
print("NOTEBOOK 06: LANDSCAPE-ORIENTED MODELLING OF URBAN SURFACE WARMING")
print("=" * 70)
print(f"\nStudy Year:          {STUDY_YEAR}")
print(f"Output Directory:    {MODEL_OUTPUT_DIR}")
print(f"Hierarchy Blocks:    {' → '.join(HIERARCHY_BLOCKS)}")
print(f"\nAnalysis 1 (OLS Clustered — Extent):")
print(f"  Method:            {OLS_CLUSTERED_CONFIG['hotspot_method']}")
print(f"  DV:                log({OLS_CLUSTERED_CONFIG['dependent_variable']})")
print(f"  Cluster:           {OLS_CLUSTERED_CONFIG['cluster_variable']}")
print(f"  α:                 {OLS_CLUSTERED_CONFIG['significance_level']}")
print(f"\nAnalysis 2 (OLS Clustered — Intensity):")
print(f"  Method:            {OLS_INTENSITY_CONFIG['hotspot_method']}")
print(f"  DV Day:            {OLS_INTENSITY_CONFIG['dependent_variable_day']}")
print(f"  DV Night:          {OLS_INTENSITY_CONFIG['dependent_variable_night']}")
print(f"  Cluster:           {OLS_INTENSITY_CONFIG['cluster_variable']}")
print(f"  α:                 {OLS_INTENSITY_CONFIG['significance_level']}")
print(f"\nWild Cluster Bootstrap:")
print(f"  Enabled:           {RUN_BOOTSTRAP}")
print(f"  Replications (B):  {N_BOOTSTRAP}")
print(f"  Weights:           Rademacher (+1/-1)")
print(f"  Random seed:       {BOOT_SEED}")
print(f"\nReference: Cameron, Gelbach & Miller (2008, Rev. Econ. Stat.)")
print("=" * 70)

<a id="section-1-2"></a>

### 1.2 Load Data Sources

In [ ]:
# ── 1. City list (from prior analysis) ──────────────────────────────
CITIES = load_cities_from_daily_analysis(STUDY_YEAR)
print(f"✓ {len(CITIES)} cities loaded: {', '.join(CITIES[:10])}{'...' if len(CITIES) > 10 else ''}")

# ── 2. Enriched hotspot polygons from NB05 ──────────────────────────
HOTSPOT_PATH = OUTPUT_DIR / '05_hotspot_analysis' / 'hotspot_polygons' / 'hotspot_polygons_EPSG4326.geojson'
hotspot_gdf = gpd.read_file(HOTSPOT_PATH)
print(f"✓ Hotspot polygons loaded: {len(hotspot_gdf)} rows, CRS = {hotspot_gdf.crs}")
print(f"  Methods: {hotspot_gdf['method'].value_counts().to_dict()}")
print(f"  Periods: {hotspot_gdf['period'].value_counts().to_dict()}")

# ── 3. AOI boundaries ───────────────────────────────────────────────
AOI_BASE = OUTPUT_DIR / '01_aoi_delineation'
aoi_gdf = load_aoi_boundaries(AOI_BASE, CITIES)

# ── 4. Landscape character polygons ─────────────────────────────────
landscape_gdf = load_landscape_data(LANDSCAPE_DATA_PATH)
print(f"✓ Landscape polygons loaded: {len(landscape_gdf)} features")
print(f"  Class columns: {[c for c in landscape_gdf.columns if 'Class' in c]}")

print(f"\n✓ All data sources loaded successfully.")

<a id="section-2"></a>

## 2. Dataset Construction

<a id="section-2-2"></a>

### 2.1 Polygon-Level Continuous Dataset (Analyses 1 & 2)

Filter to P90 hotspot polygons (Day / Night separately), using the enriched landscape attributes already present in the hotspot GeoDataFrame (`*_mode` columns from NB05).  This dataset serves both Analysis 1 (extent) and Analysis 2 (intensity).

In [ ]:
# ── Extract dominant landscape classes per city AOI ──────────────────
print("Extracting dominant landscape classes for each city AOI...")
landscape_classes = extract_dominant_landscape(
    aoi_gdf=aoi_gdf,
    landscape_gdf=landscape_gdf,
    class_columns=HIERARCHY_BLOCKS,
)
print(f"✓ Landscape classes extracted for {len(landscape_classes)} cities")
n_cities = len(landscape_classes)
print(f"\n✓ {n_cities} cities available for modelling.")

<a id="section-2-3"></a>

### 2.2 Data Composition — Donut Chart (Day | Night)

A nested donut visualises how the P90 hotspot polygons are distributed across the three landscape hierarchies.  
**Rings (inside → outside):** Climate Zone → Geology → Elevation.  Centre label shows the total polygon count for each period.

In [ ]:
# ── Filter P90 hotspot polygons ─────────────────────────────────────
ols_method = OLS_CLUSTERED_CONFIG['hotspot_method']
dv_raw = OLS_CLUSTERED_CONFIG['dependent_variable']

p90_gdf = hotspot_gdf[hotspot_gdf['method'] == ols_method].copy()
print(f"P90 polygons: {len(p90_gdf)} total")

# ── Rename _mode columns for clean modelling ────────────────────────
# NB05 enrichment uses Climate_Class_mode, etc.  Rename to plain names.
rename_map = {f'{col}_mode': col for col in HIERARCHY_BLOCKS}
for old_name, new_name in rename_map.items():
    if old_name in p90_gdf.columns:
        p90_gdf[new_name] = p90_gdf[old_name]
        print(f"  Mapped: {old_name} → {new_name}")

# ── Drop rows with missing class values ─────────────────────────────
before = len(p90_gdf)
p90_gdf = p90_gdf.dropna(subset=HIERARCHY_BLOCKS)
after = len(p90_gdf)
if before != after:
    print(f"  ⚠ Dropped {before - after} rows with missing landscape classes")

# ── Log-transform the extent DV (Analysis 2) ────────────────────────
raw_skew = p90_gdf[dv_raw].skew()
skew_threshold = OLS_CLUSTERED_CONFIG['skewness_threshold']

if abs(raw_skew) > skew_threshold:
    p90_gdf['log_area_km2'] = np.log(p90_gdf[dv_raw])
    dv_col = 'log_area_km2'
    log_skew = p90_gdf[dv_col].skew()
    print(f"\n✓ Log-transform applied (area): skewness {raw_skew:.2f} → {log_skew:.2f}")
    print(f"  (threshold = {skew_threshold})")
else:
    dv_col = dv_raw
    print(f"\n  No transform needed (area): skewness = {raw_skew:.2f} (< {skew_threshold})")

# ── Check intensity DVs (Analysis 3) ────────────────────────────────
dv_int_day = OLS_INTENSITY_CONFIG['dependent_variable_day']
dv_int_night = OLS_INTENSITY_CONFIG['dependent_variable_night']
int_skew_threshold = OLS_INTENSITY_CONFIG['skewness_threshold']

for period, dv_int in [('Day', dv_int_day), ('Night', dv_int_night)]:
    subset = p90_gdf[p90_gdf['period'] == period]
    int_skew = subset[dv_int].skew()
    if abs(int_skew) > int_skew_threshold:
        log_col = f'log_{dv_int}'
        p90_gdf.loc[p90_gdf['period'] == period, log_col] = np.log(subset[dv_int])
        log_skew = p90_gdf.loc[p90_gdf['period'] == period, log_col].skew()
        print(f"\n✓ Log-transform applied ({dv_int}): skewness {int_skew:.2f} → {log_skew:.2f}")
    else:
        print(f"\n  No transform needed ({dv_int}): skewness = {int_skew:.2f} (< {int_skew_threshold})")

# ── Per-period summary ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("CONTINUOUS DATASET SUMMARY")
print("=" * 60)
for period in OLS_CLUSTERED_CONFIG['periods']:
    subset = p90_gdf[p90_gdf['period'] == period]
    n_cities = subset['city'].nunique()
    print(f"\n  {period}: {len(subset)} polygons across {n_cities} cities")
    print(f"    {dv_col}: mean={subset[dv_col].mean():.3f}, std={subset[dv_col].std():.3f}")
    dv_int = dv_int_day if period == 'Day' else dv_int_night
    print(f"    {dv_int}: mean={subset[dv_int].mean():.2f}°C, std={subset[dv_int].std():.2f}")

print("\n--- Class distributions ---")
for col in HIERARCHY_BLOCKS:
    print(f"\n  {col}:")
    print(f"    {p90_gdf[col].value_counts().to_dict()}")

<a id="section-2-4"></a>

### 2.3 Data Composition — Sankey Diagram (Day / Night)

Three-column alluvial plot: **Climate → Geology → Elevation**, with separate panels for Day and Night.  Band width = number of P90 polygons flowing through that landscape combination.

In [ ]:
# ── Data Composition Donut Chart (Day | Night) ─────────────────────
fig_donut = plot_data_composition_donut(
    df=p90_gdf,
    block_columns=HIERARCHY_BLOCKS,
    periods=['Day', 'Night'],
    title='Data Composition: P90 Hotspot Polygons by Landscape Class',
    output_path=FIGURES_DIR / 'data_composition_donut.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()

<a id="section-2-4"></a>

### 2.4 Data Composition — Sankey Diagram (Day / Night)

A three-column Sankey (alluvial) diagram reveals how individual P90 hotspot polygons flow through the landscape hierarchy.  
**Columns (left → right):** Climate Zone → Geology → Elevation.  Band widths are proportional to polygon counts; colour follows the source node.  Day and Night are shown as separate panels.

In [ ]:
# ── Data Composition Sankey Diagram (Day / Night) ──────────────────
fig_sankey = plot_data_composition_sankey(
    df=p90_gdf,
    block_columns=HIERARCHY_BLOCKS,
    periods=['Day', 'Night'],
    title='Data Composition: Sankey — Climate → Geology → Elevation',
    output_path=FIGURES_DIR / 'data_composition_sankey.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()

<a id="section-2-5"></a>

### 2.4 Skewness Diagnostic — Hotspot Extent (Area)

2×2 panel: Day/Night × Raw/Log. A red dashed normal-distribution bell curve is overlaid for comparison. If skewness exceeds the threshold (1.0), the log-transform is applied automatically.

In [ ]:
# ── Skewness diagnostic: Area (Day | Night) ─────────────────────────
p90_day_area = p90_gdf.loc[p90_gdf['period'] == 'Day', dv_raw]
p90_night_area = p90_gdf.loc[p90_gdf['period'] == 'Night', dv_raw]

fig_skew_area, skew_stats_area = plot_skewness_diagnostic(
    day_series=p90_day_area,
    night_series=p90_night_area,
    variable_name='area_km2',
    output_path=FIGURES_DIR / 'skewness_diagnostic_area.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
print(f"\nDay  — raw skewness: {skew_stats_area['day_raw_skew']:.3f},  log skewness: {skew_stats_area['day_log_skew']:.3f}")
print(f"Night — raw skewness: {skew_stats_area['night_raw_skew']:.3f},  log skewness: {skew_stats_area['night_log_skew']:.3f}")
plt.show()

<a id="section-2-6"></a>

### 2.5 Skewness Diagnostic — Hotspot Intensity (Mean LST)

Same layout as the area diagnostic, but for the mean LST within each hotspot polygon.

In [ ]:
# ── Skewness diagnostic: Mean LST (Day | Night) ─────────────────────
p90_day_lst = p90_gdf.loc[p90_gdf['period'] == 'Day', dv_int_day]
p90_night_lst = p90_gdf.loc[p90_gdf['period'] == 'Night', dv_int_night]

fig_skew_lst, skew_stats_lst = plot_skewness_diagnostic(
    day_series=p90_day_lst,
    night_series=p90_night_lst,
    variable_name='Mean LST (°C)',
    output_path=FIGURES_DIR / 'skewness_diagnostic_mean_lst.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
print(f"\nDay  — raw skewness: {skew_stats_lst['day_raw_skew']:.3f},  log skewness: {skew_stats_lst['day_log_skew']:.3f}")
print(f"Night — raw skewness: {skew_stats_lst['night_raw_skew']:.3f},  log skewness: {skew_stats_lst['night_log_skew']:.3f}")
plt.show()

<a id="section-3"></a>

## 3. Analysis 1 — Hotspot Extent (OLS with Clustered SEs)

Hierarchical OLS regression with city-clustered standard errors, fitted separately for Day and Night.  
The dependent variable is $\ln(\text{area}_{\text{km}^2})$ to address right skew.

**Hypothesis:** Physiographic landscape elements (Climate → Geology → Elevation) significantly predict the spatial extent of P90 hotspot polygons.

<a id="section-3-1"></a>

### 3.1 Day & Night Models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — DAY
# ═══════════════════════════════════════════════════════════════════════
p90_day = p90_gdf[p90_gdf['period'] == 'Day'].copy()
print("=" * 70)
print("ANALYSIS 1 — DAY: Hierarchical OLS with Clustered SEs (Extent)")
print("=" * 70)
print(f"DV: {dv_col} | n = {len(p90_day)} polygons | {p90_day['city'].nunique()} cities")
print(f"Hierarchy: {' → '.join(HIERARCHY_BLOCKS)}")

ols_day = fit_hierarchical_ols_clustered(
    df=p90_day,
    dv_column=dv_col,
    block_columns=HIERARCHY_BLOCKS,
    cluster_column=OLS_CLUSTERED_CONFIG['cluster_variable'],
    alpha=OLS_CLUSTERED_CONFIG['significance_level'],
    verbose=True,
    run_bootstrap=RUN_BOOTSTRAP,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOT_SEED,
)

print("\n" + "─" * 60)
print("MODEL COMPARISON TABLE")
print("─" * 60)
display(ols_day['comparison'])

if ols_day['block_tests']:
    print("\n" + "─" * 60)
    print("BLOCK-LEVEL PARTIAL F-TESTS")
    print("─" * 60)
    for test in ols_day['block_tests']:
        sig = '***' if test['p_value'] < 0.001 else '**' if test['p_value'] < 0.01 else '*' if test['significant'] else 'ns'
        print(f"  + {test['block_added']:20s}  F = {test['f_statistic']:8.3f}  df = ({test['df_num']},{test['df_denom']})  p = {test['p_value']:.4f}  {sig}")

# ── Full model coefficients (with Wild Bootstrap p-values) ──────────
full_ols_day = ols_day['models'][-1]
boot_ext_day_df = ols_day.get('bootstrap', None)
coef_day = format_ols_coefficients(
    full_ols_day,
    alpha=OLS_CLUSTERED_CONFIG['significance_level'],
    log_transformed=(dv_col != dv_raw),
    bootstrap_df=boot_ext_day_df,
)
print("\n" + "─" * 60)
print("FULL MODEL — COEFFICIENTS (Clustered SEs + Wild Bootstrap)")
print("─" * 60)
display(coef_day)

coef_day.to_csv(TABLES_DIR / 'analysis1_ols_day_coefficients.csv', index=False)
ols_day['comparison'].to_csv(TABLES_DIR / 'analysis1_ols_day_comparison.csv', index=False)

# ═══════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — NIGHT
# ═══════════════════════════════════════════════════════════════════════
p90_night = p90_gdf[p90_gdf['period'] == 'Night'].copy()
print("\n" + "=" * 70)
print("ANALYSIS 1 — NIGHT: Hierarchical OLS with Clustered SEs (Extent)")
print("=" * 70)
print(f"DV: {dv_col} | n = {len(p90_night)} polygons | {p90_night['city'].nunique()} cities")
print(f"Hierarchy: {' → '.join(HIERARCHY_BLOCKS)}")

ols_night = fit_hierarchical_ols_clustered(
    df=p90_night,
    dv_column=dv_col,
    block_columns=HIERARCHY_BLOCKS,
    cluster_column=OLS_CLUSTERED_CONFIG['cluster_variable'],
    alpha=OLS_CLUSTERED_CONFIG['significance_level'],
    verbose=True,
    run_bootstrap=RUN_BOOTSTRAP,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOT_SEED,
)

print("\n" + "─" * 60)
print("MODEL COMPARISON TABLE")
print("─" * 60)
display(ols_night['comparison'])

if ols_night['block_tests']:
    print("\n" + "─" * 60)
    print("BLOCK-LEVEL PARTIAL F-TESTS")
    print("─" * 60)
    for test in ols_night['block_tests']:
        sig = '***' if test['p_value'] < 0.001 else '**' if test['p_value'] < 0.01 else '*' if test['significant'] else 'ns'
        print(f"  + {test['block_added']:20s}  F = {test['f_statistic']:8.3f}  df = ({test['df_num']},{test['df_denom']})  p = {test['p_value']:.4f}  {sig}")

# ── Full model coefficients (with Wild Bootstrap p-values) ──────────
full_ols_night = ols_night['models'][-1]
boot_ext_night_df = ols_night.get('bootstrap', None)
coef_night = format_ols_coefficients(
    full_ols_night,
    alpha=OLS_CLUSTERED_CONFIG['significance_level'],
    log_transformed=(dv_col != dv_raw),
    bootstrap_df=boot_ext_night_df,
)
print("\n" + "─" * 60)
print("FULL MODEL — COEFFICIENTS (Clustered SEs + Wild Bootstrap)")
print("─" * 60)
display(coef_night)

coef_night.to_csv(TABLES_DIR / 'analysis1_ols_night_coefficients.csv', index=False)
ols_night['comparison'].to_csv(TABLES_DIR / 'analysis1_ols_night_comparison.csv', index=False)
print(f"\n✓ All Analysis 1 tables saved to {TABLES_DIR.name}/")

<a id="section-3-2"></a>

### 3.2 Variance Partitioning — OLS Extent (Day | Night)

In [ ]:
fig_vp_ols = plot_variance_partitioning_panel(
    comparison_day=ols_day['comparison'],
    comparison_night=ols_night['comparison'],
    r2_col='R²',
    delta_col='ΔR²',
    title='Analysis 1 — Variance Partitioning (OLS Extent): Day | Night',
    ylabel='R²',
    block_columns=HIERARCHY_BLOCKS,
    output_path=FIGURES_DIR / 'analysis1_ols_variance_partitioning_panel.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()
print("\n✓ Variance partitioning panel saved.")

<a id="section-3-3"></a>

### 3.3 Coefficient Forest Plots — OLS Extent (Day | Night)

In [ ]:
fig_forest_ols = plot_coefficient_forest_panel(
    coef_day=coef_day,
    coef_night=coef_night,
    title='Analysis 1 — Coefficient Forest Plot (OLS Extent): Day | Night',
    output_path=FIGURES_DIR / 'analysis1_ols_forest_plot_panel.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()
print("\n✓ Forest plot panel saved.")

<a id="section-4"></a>

## 4. Analysis 2 — Hotspot Intensity (OLS with Clustered SEs)

Hierarchical OLS regression predicting the **mean LST (°C)** within each P90 hotspot polygon.  This analysis examines whether landscape elements explain not just *where* hotspots occur or *how large* they are, but *how intense* they are in terms of absolute surface temperature.

- Day DV: `LST_day_mean` | Night DV: `LST_night_mean`
- Clustering: city-level clustered standard errors

<a id="section-4-1"></a>

### 4.1 Day & Night Models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — DAY: Hotspot Intensity
# ═══════════════════════════════════════════════════════════════════════
# Determine DV (with or without log-transform)
int_day_skew = p90_day[dv_int_day].skew()
if abs(int_day_skew) > int_skew_threshold:
    dv_int_day_col = f'log_{dv_int_day}'
    if dv_int_day_col not in p90_day.columns:
        p90_day[dv_int_day_col] = np.log(p90_day[dv_int_day])
    int_log_transformed_day = True
else:
    dv_int_day_col = dv_int_day
    int_log_transformed_day = False

print("=" * 70)
print("ANALYSIS 2 — DAY: Hotspot Intensity (OLS Clustered)")
print("=" * 70)
print(f"DV: {dv_int_day_col} | n = {len(p90_day)} polygons | {p90_day['city'].nunique()} cities")
print(f"Hierarchy: {' → '.join(HIERARCHY_BLOCKS)}")

ols_int_day = fit_hierarchical_ols_clustered(
    df=p90_day,
    dv_column=dv_int_day_col,
    block_columns=HIERARCHY_BLOCKS,
    cluster_column=OLS_INTENSITY_CONFIG['cluster_variable'],
    alpha=OLS_INTENSITY_CONFIG['significance_level'],
    verbose=True,
    run_bootstrap=RUN_BOOTSTRAP,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOT_SEED,
)

print("\n" + "─" * 60)
print("MODEL COMPARISON TABLE")
print("─" * 60)
display(ols_int_day['comparison'])

if ols_int_day['block_tests']:
    print("\n" + "─" * 60)
    print("BLOCK-LEVEL PARTIAL F-TESTS")
    print("─" * 60)
    for test in ols_int_day['block_tests']:
        sig = '***' if test['p_value'] < 0.001 else '**' if test['p_value'] < 0.01 else '*' if test['significant'] else 'ns'
        print(f"  + {test['block_added']:20s}  F = {test['f_statistic']:8.3f}  df = ({test['df_num']},{test['df_denom']})  p = {test['p_value']:.4f}  {sig}")

full_ols_int_day = ols_int_day['models'][-1]
boot_int_day_df = ols_int_day.get('bootstrap', None)
coef_int_day = format_ols_coefficients(
    full_ols_int_day,
    alpha=OLS_INTENSITY_CONFIG['significance_level'],
    log_transformed=int_log_transformed_day,
    bootstrap_df=boot_int_day_df,
)
print("\n" + "─" * 60)
print("FULL MODEL — COEFFICIENTS (Clustered SEs + Wild Bootstrap)")
print("─" * 60)
display(coef_int_day)

coef_int_day.to_csv(TABLES_DIR / 'analysis2_ols_intensity_day_coefficients.csv', index=False)
ols_int_day['comparison'].to_csv(TABLES_DIR / 'analysis2_ols_intensity_day_comparison.csv', index=False)

# ═══════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — NIGHT: Hotspot Intensity
# ═══════════════════════════════════════════════════════════════════════
int_night_skew = p90_night[dv_int_night].skew()
if abs(int_night_skew) > int_skew_threshold:
    dv_int_night_col = f'log_{dv_int_night}'
    if dv_int_night_col not in p90_night.columns:
        p90_night[dv_int_night_col] = np.log(p90_night[dv_int_night])
    int_log_transformed_night = True
else:
    dv_int_night_col = dv_int_night
    int_log_transformed_night = False

print("\n" + "=" * 70)
print("ANALYSIS 2 — NIGHT: Hotspot Intensity (OLS Clustered)")
print("=" * 70)
print(f"DV: {dv_int_night_col} | n = {len(p90_night)} polygons | {p90_night['city'].nunique()} cities")
print(f"Hierarchy: {' → '.join(HIERARCHY_BLOCKS)}")

ols_int_night = fit_hierarchical_ols_clustered(
    df=p90_night,
    dv_column=dv_int_night_col,
    block_columns=HIERARCHY_BLOCKS,
    cluster_column=OLS_INTENSITY_CONFIG['cluster_variable'],
    alpha=OLS_INTENSITY_CONFIG['significance_level'],
    verbose=True,
    run_bootstrap=RUN_BOOTSTRAP,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOT_SEED,
)

print("\n" + "─" * 60)
print("MODEL COMPARISON TABLE")
print("─" * 60)
display(ols_int_night['comparison'])

if ols_int_night['block_tests']:
    print("\n" + "─" * 60)
    print("BLOCK-LEVEL PARTIAL F-TESTS")
    print("─" * 60)
    for test in ols_int_night['block_tests']:
        sig = '***' if test['p_value'] < 0.001 else '**' if test['p_value'] < 0.01 else '*' if test['significant'] else 'ns'
        print(f"  + {test['block_added']:20s}  F = {test['f_statistic']:8.3f}  df = ({test['df_num']},{test['df_denom']})  p = {test['p_value']:.4f}  {sig}")

full_ols_int_night = ols_int_night['models'][-1]
boot_int_night_df = ols_int_night.get('bootstrap', None)
coef_int_night = format_ols_coefficients(
    full_ols_int_night,
    alpha=OLS_INTENSITY_CONFIG['significance_level'],
    log_transformed=int_log_transformed_night,
    bootstrap_df=boot_int_night_df,
)
print("\n" + "─" * 60)
print("FULL MODEL — COEFFICIENTS (Clustered SEs + Wild Bootstrap)")
print("─" * 60)
display(coef_int_night)

coef_int_night.to_csv(TABLES_DIR / 'analysis2_ols_intensity_night_coefficients.csv', index=False)
ols_int_night['comparison'].to_csv(TABLES_DIR / 'analysis2_ols_intensity_night_comparison.csv', index=False)
print(f"\n✓ All Analysis 2 tables saved to {TABLES_DIR.name}/")

<a id="section-4-2"></a>

### 4.2 Variance Partitioning — OLS Intensity (Day | Night)

In [ ]:
fig_vp_int = plot_variance_partitioning_panel(
    comparison_day=ols_int_day['comparison'],
    comparison_night=ols_int_night['comparison'],
    r2_col='R²',
    delta_col='ΔR²',
    title='Analysis 2 — Variance Partitioning (OLS Intensity): Day | Night',
    ylabel='R²',
    block_columns=HIERARCHY_BLOCKS,
    output_path=FIGURES_DIR / 'analysis2_ols_intensity_variance_partitioning_panel.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()
print("\n✓ Variance partitioning panel saved.")

<a id="section-4-3"></a>

### 4.3 Coefficient Forest Plots — OLS Intensity (Day | Night)

In [ ]:
fig_forest_int = plot_coefficient_forest_panel(
    coef_day=coef_int_day,
    coef_night=coef_int_night,
    title='Analysis 2 — Coefficient Forest Plot (OLS Intensity): Day | Night',
    output_path=FIGURES_DIR / 'analysis2_ols_intensity_forest_plot_panel.png',
    dpi=MODEL_EXPORT_CONFIG['figure_dpi'],
    show=True,
)
plt.show()
print("\n✓ Forest plot panel saved.")

<a id="section-5"></a>

## 5. Summary and Export

Consolidate results, save metadata, and print a final summary.

In [ ]:
# ── Save model metadata (JSON) ──────────────────────────────────────
for period, results in [('Day', ols_day), ('Night', ols_night)]:
    save_model_metadata(
        output_dir=TABLES_DIR,
        analysis_name='analysis1_ols_extent',
        period=period,
        comparison_df=results['comparison'],
        block_tests=results['block_tests'],
        config_dict={**OLS_CLUSTERED_CONFIG, **WILD_BOOTSTRAP_CONFIG},
    )

for period, results in [('Day', ols_int_day), ('Night', ols_int_night)]:
    save_model_metadata(
        output_dir=TABLES_DIR,
        analysis_name='analysis2_ols_intensity',
        period=period,
        comparison_df=results['comparison'],
        block_tests=results['block_tests'],
        config_dict={**OLS_INTENSITY_CONFIG, **WILD_BOOTSTRAP_CONFIG},
    )

# ── Final summary ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("NOTEBOOK 06 — FINAL SUMMARY")
print("=" * 70)

print("\n─── Analysis 1: Hotspot Extent (OLS Clustered, P90) ───")
for label, results in [('Day', ols_day), ('Night', ols_night)]:
    comp = results['comparison']
    full_r2 = comp['R²'].iloc[-1]
    adj_r2 = comp['Adj-R²'].iloc[-1]
    n_sig_blocks = sum(1 for t in results['block_tests'] if t.get('significant', False))
    print(f"  {label}: R² = {full_r2:.4f}, Adj-R² = {adj_r2:.4f}  |  {n_sig_blocks}/{len(results['block_tests'])} blocks significant")
    for idx, row in comp.iterrows():
        print(f"    Block {row['Block']}: {row['Variables']}  →  ΔR² = {row['ΔR²']:.4f}")
    if results.get('bootstrap') is not None:
        boot_sig = results['bootstrap'][results['bootstrap']['Boot_p'] < 0.05]
        print(f"  Bootstrap: {len(boot_sig)} / {len(results['bootstrap'])} variables significant at p < 0.05")

print("\n─── Analysis 2: Hotspot Intensity (OLS Clustered, P90 Mean LST) ───")
for label, results in [('Day', ols_int_day), ('Night', ols_int_night)]:
    comp = results['comparison']
    full_r2 = comp['R²'].iloc[-1]
    adj_r2 = comp['Adj-R²'].iloc[-1]
    n_sig_blocks = sum(1 for t in results['block_tests'] if t.get('significant', False))
    print(f"  {label}: R² = {full_r2:.4f}, Adj-R² = {adj_r2:.4f}  |  {n_sig_blocks}/{len(results['block_tests'])} blocks significant")
    for idx, row in comp.iterrows():
        print(f"    Block {row['Block']}: {row['Variables']}  →  ΔR² = {row['ΔR²']:.4f}")
    if results.get('bootstrap') is not None:
        boot_sig = results['bootstrap'][results['bootstrap']['Boot_p'] < 0.05]
        print(f"  Bootstrap: {len(boot_sig)} / {len(results['bootstrap'])} variables significant at p < 0.05")

# ── List saved outputs ──────────────────────────────────────────────
print("\n─── Saved Outputs ───")
print(f"\n  Tables ({TABLES_DIR}):")
for f in sorted(TABLES_DIR.glob('*')):
    print(f"    • {f.name}")

print(f"\n  Figures ({FIGURES_DIR}):")
for f in sorted(FIGURES_DIR.glob('*')):
    print(f"    • {f.name}")

print("\n" + "=" * 70)
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)
print("\n✓ Notebook 06 complete.")